In [0]:
# Silver Layer: Products Dimension (SCD Type 1 BATCH ITERATOR)

from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql.functions import (
    col, lower, trim, to_timestamp, when,
    current_timestamp, regexp_replace, lit, row_number, desc, upper, try_to_date
)
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType

spark = SparkSession.builder.getOrCreate()

BRONZE_TABLE = "ecomstore.ecomlake.bronze_products"
SILVER_TABLE = "ecomstore.ecomlake.silver_products"
QUARANTINE_TABLE = "ecomstore.ecomlake.silver_quarantine_products"
TEMP_STAGING_TABLE = "ecomstore.ecomlake.temp_silver_products_staging"

bronze_products = spark.read.table(BRONZE_TABLE)

# 1. Fetch batches chronologically
batches_df = spark.sql(f"SELECT DISTINCT _batch_id FROM {BRONZE_TABLE} ORDER BY _batch_id").collect()
batches = [row['_batch_id'] for row in batches_df]

for current_batch in batches:
    print(f"\n🚀 Processing Batch: {current_batch}")
    
    incoming_batch = bronze_products.filter(col("_batch_id") == current_batch)

    # 2. Deduplicate WITHIN the current batch
    w = Window.partitionBy("product_id").orderBy(desc("updated_at"), desc("_ingestion_timestamp"))
    deduped = incoming_batch.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn")

    # 3. Cleanse
    clean_products = (
        deduped
        .withColumn("product_id", trim(upper(col("product_id"))))
        .withColumn("category", trim(col("category")))
        .withColumn("brand", lower(trim(col("brand"))))
        .withColumn("selling_price", regexp_replace(col("selling_price"), r"[^\d.]", "").cast(DoubleType()))
        .withColumn("selling_price", when(col("selling_price") < 0, None).otherwise(col("selling_price")))
        .withColumn("mrp", col("mrp").cast(DoubleType()))
        .withColumn("is_active",
            when(col("is_active").isin("true", "1", "True"), True)
            .when(col("is_active").isin("false", "0", "False"), False)
            .otherwise(None)
        )
        .withColumn("launch_date", try_to_date(col("launch_date")))
        .withColumn("updated_at", to_timestamp(col("updated_at")))
        .withColumn("_silver_processed_at", current_timestamp())
        .select("product_id", "product_name", "category", "sub_category", "brand",
                "seller_id", "mrp", "selling_price", "cost_price",
                "weight_kg", "is_active", "launch_date", "updated_at",
                "_batch_id", "_silver_processed_at")
    )

    # Disk Staging
    clean_products.coalesce(1).write.format("delta").mode("overwrite").saveAsTable(TEMP_STAGING_TABLE)
    staged_products = spark.read.table(TEMP_STAGING_TABLE)

    # 4. Quarantine Routing
    good_records = staged_products.filter(col("product_id").isNotNull() & col("selling_price").isNotNull())
    bad_records = staged_products.filter(col("product_id").isNull() | col("selling_price").isNull())

    if bad_records.count() > 0:
        (
            bad_records
            .withColumn("rejection_reason", when(col("product_id").isNull(), lit("null_product_id")).otherwise(lit("null_selling_price")))
            .coalesce(1)
            .write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(QUARANTINE_TABLE)
        )

    # 5. SCD TYPE 1 MERGE
    if spark.catalog.tableExists(SILVER_TABLE):
        silver_products = DeltaTable.forName(spark, SILVER_TABLE)
        (
            silver_products.alias("target")
            .merge(good_records.alias("source"), "target.product_id = source.product_id")
            # Overwrite all columns when matched (SCD Type 1 behavior)
            .whenMatchedUpdate(
                condition="source.updated_at > target.updated_at",
                set={"target." + c: "source." + c for c in clean_products.columns if c != "product_id"}
            )
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"✅ SCD Type 1 MERGE complete for Batch: {current_batch}")
    else:
        good_records.coalesce(1).write.format("delta").mode("overwrite").option("delta.autoOptimize.optimizeWrite", "true").option("delta.autoOptimize.autoCompact", "true").saveAsTable(SILVER_TABLE)
        print(f"✅ Created initial table for Batch: {current_batch}")

# Clean up staging table
spark.sql(f"DROP TABLE IF EXISTS {TEMP_STAGING_TABLE}")